In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, make_scorer
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor, StackingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.multioutput import MultiOutputRegressor
import xgboost as xgb
import lightgbm as lgb
from scipy import stats
import shap
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 1. 데이터 로드 
df_merged = pd.read_csv('df_merged.csv')
X_linear = pd.read_csv('X_preprocessed_linear.csv')
X_nn = pd.read_csv('X_preprocessed_nn.csv')
X_tree = pd.read_csv('X_preprocessed_tree.csv')
y_targets = pd.read_csv('y_targets.csv')

In [3]:
# 타겟 및 특성 이름
target_names = y_targets.columns.tolist()
feature_names = X_tree.columns.tolist()

In [4]:
# 2. 커스텀 평가 함수
def calculate_all_metrics(y_true, y_pred):
    """모든 요구된 평가 지표 계산"""
    metrics = {}
    
    # 기본 회귀 지표
    metrics['MAE'] = mean_absolute_error(y_true, y_pred)
    metrics['MSE'] = mean_squared_error(y_true, y_pred)
    metrics['RMSE'] = np.sqrt(metrics['MSE'])
    
    # MSLE, RMSLE
    try:
        y_true_pos = np.clip(y_true, a_min=1e-10, a_max=None)
        y_pred_pos = np.clip(y_pred, a_min=1e-10, a_max=None)
        metrics['MSLE'] = mean_squared_log_error(y_true_pos, y_pred_pos)
        metrics['RMSLE'] = np.sqrt(metrics['MSLE'])
    except:
        metrics['MSLE'] = np.nan
        metrics['RMSLE'] = np.nan
    
    # R² 및 상관계수
    metrics['R2'] = r2_score(y_true, y_pred)
    
    # 다중 출력의 경우 평균 상관계수
    if len(y_true.shape) > 1:
        correlations = []
        for i in range(y_true.shape[1]):
            if np.std(y_true[:, i]) > 0 and np.std(y_pred[:, i]) > 0:
                corr = np.corrcoef(y_true[:, i], y_pred[:, i])[0, 1]
                correlations.append(corr)
        metrics['R'] = np.mean(correlations) if correlations else 0
    else:
        if np.std(y_true) > 0 and np.std(y_pred) > 0:
            metrics['R'] = np.corrcoef(y_true.flatten(), y_pred.flatten())[0, 1]
        else:
            metrics['R'] = 0
    
    return metrics

In [5]:
# Scikit-learn용 scorer 생성
def msle_scorer(y_true, y_pred):
    y_true_pos = np.clip(y_true, a_min=1e-10, a_max=None)
    y_pred_pos = np.clip(y_pred, a_min=1e-10, a_max=None)
    return mean_squared_log_error(y_true_pos, y_pred_pos)

In [6]:
scoring_dict = {
    'mae': 'neg_mean_absolute_error',
    'mse': 'neg_mean_squared_error',
    'r2': 'r2',
    'msle': make_scorer(msle_scorer, greater_is_better=False)
}

In [7]:
# 3. 시계열 교차검증 전략
# 시계열 분할 (월 기준)
months = df_merged['month'].values
n_splits = 3

In [8]:
# 4. 앙상블 모델 구축
# 기본 모델 정의
base_models = {
    'rf': RandomForestRegressor(
        n_estimators=200,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42
    ),
    'xgb': MultiOutputRegressor(xgb.XGBRegressor(
        n_estimators=200,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )),
    'lgb': MultiOutputRegressor(lgb.LGBMRegressor(
        n_estimators=200,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbose=-1
    )),
    'gbr': GradientBoostingRegressor(
        n_estimators=150,
        max_depth=6,
        learning_rate=0.08,
        subsample=0.8,
        random_state=42
    )
}

In [9]:
# 메타 모델
meta_model = Ridge(alpha=1.0)

In [10]:
# 5. 교차검증 수행
# 시계열 분할을 위한 커스텀 함수
def time_series_cv(X, y, months, n_splits=3):
    """월 기준 시계열 교차검증"""
    cv_results = []
    unique_months = np.sort(np.unique(months))
    
    for i in range(n_splits):
        train_months = unique_months[:i+2]  # 5월부터 시작
        val_month = unique_months[i+2]
        
        train_idx = np.where(np.isin(months, train_months))[0]
        val_idx = np.where(months == val_month)[0]
        
        cv_results.append((train_idx, val_idx))
    
    return cv_results

In [11]:
# 각 모델별 교차검증
cv_splits = time_series_cv(X_tree.values, y_targets.values, months, n_splits)
model_cv_results = {}

for name, model in base_models.items():
    print(f"\n교차검증: {name}")
    cv_scores = {'mae': [], 'mse': [], 'r2': []}
    
    for fold, (train_idx, val_idx) in enumerate(cv_splits):
        # 데이터 분할
        X_train_cv = X_tree.values[train_idx]
        y_train_cv = y_targets.values[train_idx]
        X_val_cv = X_tree.values[val_idx]
        y_val_cv = y_targets.values[val_idx]
        
        # 모델 학습 및 예측
        if name in ['xgb', 'lgb']:
            model_copy = model
        else:
            model_copy = MultiOutputRegressor(model)
        
        model_copy.fit(X_train_cv, y_train_cv)
        y_pred_cv = model_copy.predict(X_val_cv)
        
        # 평가
        metrics = calculate_all_metrics(y_val_cv, y_pred_cv)
        cv_scores['mae'].append(metrics['MAE'])
        cv_scores['mse'].append(metrics['MSE'])
        cv_scores['r2'].append(metrics['R2'])
        
    # 평균 점수 저장
    model_cv_results[name] = {
        'MAE': np.mean(cv_scores['mae']),
        'MAE_std': np.std(cv_scores['mae']),
        'MSE': np.mean(cv_scores['mse']),
        'MSE_std': np.std(cv_scores['mse']),
        'R2': np.mean(cv_scores['r2']),
        'R2_std': np.std(cv_scores['r2'])
    }
    
    print(f"  평균 MAE: {model_cv_results[name]['MAE']:.4f} (±{model_cv_results[name]['MAE_std']:.4f})")
    print(f"  평균 R²: {model_cv_results[name]['R2']:.4f} (±{model_cv_results[name]['R2_std']:.4f})")


교차검증: rf
  평균 MAE: 1.1099 (±0.5662)
  평균 R²: -50.5743 (±52.9975)

교차검증: xgb
  평균 MAE: 0.8631 (±0.5477)
  평균 R²: -44.3045 (±58.7988)

교차검증: lgb
  평균 MAE: 0.8741 (±0.5420)
  평균 R²: -44.0571 (±58.0897)

교차검증: gbr
  평균 MAE: 1.0210 (±0.5315)
  평균 R²: -46.0876 (±51.8049)


In [16]:
# 6. 스태킹 앙상블 모델
# Train/Test 분할 (최종 평가용)
train_mask = months <= 8
test_mask = months == 9

X_train = X_tree.values[train_mask]
y_train = y_targets.values[train_mask]
X_test = X_tree.values[test_mask]
y_test = y_targets.values[test_mask]

# 단일-타깃 기본 모델 구성
rf_single  = base_models['rf']                                 # 이미 단일 타깃 [file:59]
xgb_single = base_models['xgb'].estimator                      # MultiOutputRegressor 내부 원본 [file:59]
lgb_single = base_models['lgb'].estimator                      # MultiOutputRegressor 내부 원본 [file:59]
meta_single = meta_model                                       # Ridge 단일 타깃 [file:59]

inner_stack = StackingRegressor(
    estimators=[('rf', rf_single), ('xgb', xgb_single), ('lgb', lgb_single)],
    final_estimator=meta_single,
    cv=3
)

stacking_model = MultiOutputRegressor(inner_stack)

print("스태킹 앙상블 학습 중...")
stacking_model.fit(X_train, y_train)  # y_train.shape == (n, 4) OK [file:59]

스태킹 앙상블 학습 중...


MultiOutputRegressor(estimator=StackingRegressor(cv=3,
                                                 estimators=[('rf',
                                                              RandomForestRegressor(max_depth=15,
                                                                                    min_samples_leaf=2,
                                                                                    min_samples_split=5,
                                                                                    n_estimators=200,
                                                                                    random_state=42)),
                                                             ('xgb',
                                                              XGBRegressor(base_score=None,
                                                                           booster=None,
                                                                           callbacks=None,
                                                                           colsample_bylevel=None,
                                                                           colsample_bynode=None,
                                                                           colsample_bytree=0.8,
                                                                           device=None,
                                                                           early_stopping_rounds=None,...
                                                                           max_cat_to_onehot=None,
                                                                           max_delta_step=None,
                                                                           max_depth=8,
                                                                           max_leaves=None,
                                                                           min_child_weight=None,
                                                                           missing=nan,
                                                                           monotone_constraints=None,
                                                                           multi_strategy=None,
                                                                           n_estimators=200,
                                                                           n_jobs=None,
                                                                           num_parallel_tree=None, ...)),
                                                             ('lgb',
                                                              LGBMRegressor(colsample_bytree=0.8,
                                                                            learning_rate=0.05,
                                                                            max_depth=8,
                                                                            n_estimators=200,
                                                                            random_state=42,
                                                                            subsample=0.8,
                                                                            verbose=-1))],
                                                 final_estimator=Ridge()))

In [17]:
# 예측 및 평가
y_pred_stack = stacking_model.predict(X_test)
stack_metrics = calculate_all_metrics(y_test, y_pred_stack)

In [18]:
print(f"\n스태킹 앙상블 성능:")
print(f"  MAE: {stack_metrics['MAE']:.4f}")
print(f"  RMSE: {stack_metrics['RMSE']:.4f}")
print(f"  R²: {stack_metrics['R2']:.4f}")


스태킹 앙상블 성능:
  MAE: 1.6695
  RMSE: 2.7636
  R²: -39.7066


In [19]:
# 7. 가중 평균 앙상블
# 개별 모델 학습 및 예측
predictions = {}
for name, model in base_models.items():
    if name in ['xgb', 'lgb']:
        model_instance = model
    else:
        model_instance = MultiOutputRegressor(model)
    
    model_instance.fit(X_train, y_train)
    predictions[name] = model_instance.predict(X_test)

In [20]:
# 가중치 최적화 (교차검증 성능 기반)
weights = {}
total_weight = 0
for name in base_models.keys():
    # R² 기반 가중치
    weight = model_cv_results[name]['R2']
    weights[name] = weight
    total_weight += weight

In [21]:
# 정규화
for name in weights.keys():
    weights[name] /= total_weight

print("모델별 가중치:")
for name, weight in weights.items():
    print(f"  {name}: {weight:.3f}")

모델별 가중치:
  rf: 0.273
  xgb: 0.239
  lgb: 0.238
  gbr: 0.249


In [22]:
# 가중 평균 예측
y_pred_weighted = np.zeros_like(y_test)
for name, pred in predictions.items():
    y_pred_weighted += weights[name] * pred

weighted_metrics = calculate_all_metrics(y_test, y_pred_weighted)

print(f"\n가중 평균 앙상블 성능:")
print(f"  MAE: {weighted_metrics['MAE']:.4f}")
print(f"  RMSE: {weighted_metrics['RMSE']:.4f}")
print(f"  R²: {weighted_metrics['R2']:.4f}")


가중 평균 앙상블 성능:
  MAE: 1.0078
  RMSE: 1.5107
  R²: -12.3149


In [23]:
# 8. 특성 중요도 분석 (SHAP)
# 최고 성능 모델 선택
best_ensemble = 'stacking' if stack_metrics['MAE'] < weighted_metrics['MAE'] else 'weighted'
print(f"\n최적 앙상블: {best_ensemble}")


최적 앙상블: weighted


In [24]:
# RandomForest 기반 특성 중요도
rf_model = MultiOutputRegressor(base_models['rf'])
rf_model.fit(X_train, y_train)

MultiOutputRegressor(estimator=RandomForestRegressor(max_depth=15,
                                                     min_samples_leaf=2,
                                                     min_samples_split=5,
                                                     n_estimators=200,
                                                     random_state=42))

In [25]:
# 각 타겟별 특성 중요도
feature_importance_by_target = {}
for i, target in enumerate(target_names):
    if hasattr(rf_model.estimators_[i], 'feature_importances_'):
        importance = rf_model.estimators_[i].feature_importances_
        feature_importance_by_target[target] = importance

In [26]:
# 평균 특성 중요도
avg_importance = np.mean(list(feature_importance_by_target.values()), axis=0)
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': avg_importance
}).sort_values('Importance', ascending=False)

print("\n핵심인자 TOP 15 (Feature Importance):")
for i, row in importance_df.head(15).iterrows():
    print(f"  {row['Feature']:30s}: {row['Importance']:.4f}")


핵심인자 TOP 15 (Feature Importance):
  stage_reproductive            : 0.2412
  month                         : 0.1919
  VPD                           : 0.0839
  stage_vegetative_early        : 0.0785
  Temp                          : 0.0780
  Chl_a                         : 0.0737
  scenario_SSP3                 : 0.0531
  TCh-Car                       : 0.0528
  Car                           : 0.0313
  Dio-RC                        : 0.0197
  stress_index                  : 0.0101
  Chl_b                         : 0.0100
  scenario_SSP5                 : 0.0087
  stage_vegetative_mid          : 0.0080
  stage_vegetative_late         : 0.0072


In [27]:
# 9. 예측 신뢰구간
# 부트스트랩을 통한 신뢰구간 추정
n_bootstrap = 100
bootstrap_predictions = []

for i in range(n_bootstrap):
    # 부트스트랩 샘플링
    indices = np.random.choice(len(X_train), size=len(X_train), replace=True)
    X_boot = X_train[indices]
    y_boot = y_train[indices]
    
    # 모델 학습 및 예측
    rf_boot = MultiOutputRegressor(RandomForestRegressor(n_estimators=50, random_state=i))
    rf_boot.fit(X_boot, y_boot)
    pred_boot = rf_boot.predict(X_test)
    bootstrap_predictions.append(pred_boot)

bootstrap_predictions = np.array(bootstrap_predictions)

# 95% 신뢰구간
lower_bound = np.percentile(bootstrap_predictions, 2.5, axis=0)
upper_bound = np.percentile(bootstrap_predictions, 97.5, axis=0)
mean_pred = np.mean(bootstrap_predictions, axis=0)

print("\n예측 신뢰구간 (95%):")
for i, target in enumerate(target_names):
    ci_width = np.mean(upper_bound[:, i] - lower_bound[:, i])
    print(f"  {target}: 평균 CI 폭 = {ci_width:.4f}")


예측 신뢰구간 (95%):
  Leaf_TPC: 평균 CI 폭 = 0.2818
  Root_TPC: 평균 CI 폭 = 0.4119
  Leaf_TFC: 평균 CI 폭 = 2.8483
  Root_TFC: 평균 CI 폭 = 0.0642


In [28]:
# 모든 모델 성능 종합
final_results = pd.DataFrame({
    'Model': ['RandomForest', 'XGBoost', 'LightGBM', 'GradientBoosting', 
              'Stacking Ensemble', 'Weighted Ensemble'],
    'MAE': [
        model_cv_results['rf']['MAE'],
        model_cv_results['xgb']['MAE'],
        model_cv_results['lgb']['MAE'],
        model_cv_results['gbr']['MAE'],
        stack_metrics['MAE'],
        weighted_metrics['MAE']
    ],
    'MSE': [
        model_cv_results['rf']['MSE'],
        model_cv_results['xgb']['MSE'],
        model_cv_results['lgb']['MSE'],
        model_cv_results['gbr']['MSE'],
        stack_metrics['MSE'],
        weighted_metrics['MSE']
    ],
    'RMSE': [
        np.sqrt(model_cv_results['rf']['MSE']),
        np.sqrt(model_cv_results['xgb']['MSE']),
        np.sqrt(model_cv_results['lgb']['MSE']),
        np.sqrt(model_cv_results['gbr']['MSE']),
        stack_metrics['RMSE'],
        weighted_metrics['RMSE']
    ],
    'R²': [
        model_cv_results['rf']['R2'],
        model_cv_results['xgb']['R2'],
        model_cv_results['lgb']['R2'],
        model_cv_results['gbr']['R2'],
        stack_metrics['R2'],
        weighted_metrics['R2']
    ]
})

print("\n모델 성능 비교:")
print(final_results.to_string(index=False))

# 최적 모델 지정
best_model_idx = final_results['MAE'].argmin()
best_model_name = final_results.loc[best_model_idx, 'Model']

print(f"\n✅ 최종 선택 모델: {best_model_name}")
print(f"   MAE: {final_results.loc[best_model_idx, 'MAE']:.4f}")
print(f"   RMSE: {final_results.loc[best_model_idx, 'RMSE']:.4f}")
print(f"   R²: {final_results.loc[best_model_idx, 'R²']:.4f}")


모델 성능 비교:
            Model      MAE      MSE     RMSE         R²
     RandomForest 1.109898 3.962893 1.990702 -50.574315
          XGBoost 0.863091 2.548646 1.596448 -44.304516
         LightGBM 0.874127 2.587766 1.608653 -44.057088
 GradientBoosting 1.021018 3.410393 1.846725 -46.087595
Stacking Ensemble 1.669487 7.637599 2.763621 -39.706601
Weighted Ensemble 1.007757 2.282173 1.510686 -12.314909

✅ 최종 선택 모델: XGBoost
   MAE: 0.8631
   RMSE: 1.5964
   R²: -44.3045


In [29]:
# 11. 시나리오별 상세 분석
scenario_info = df_merged['scenario'].values
scenario_test = scenario_info[test_mask]

In [30]:
# 최적 모델 예측 사용
if best_ensemble == 'stacking':
    y_pred_final = y_pred_stack
else:
    y_pred_final = y_pred_weighted

In [31]:
# 시나리오별 성능
for scenario in ['SSP1', 'SSP3', 'SSP5']:
    mask = scenario_test == scenario
    if mask.sum() > 0:
        metrics = calculate_all_metrics(y_test[mask], y_pred_final[mask])
        print(f"\n{scenario}:")
        print(f"  샘플 수: {mask.sum()}")
        print(f"  MAE: {metrics['MAE']:.4f}")
        print(f"  RMSE: {metrics['RMSE']:.4f}")
        print(f"  R²: {metrics['R2']:.4f}")
        print(f"  R: {metrics['R']:.4f}")


SSP1:
  샘플 수: 28
  MAE: 1.1175
  RMSE: 1.7093
  R²: -167250.2639
  R: 0.0703

SSP3:
  샘플 수: 28
  MAE: 0.8904
  RMSE: 1.1348
  R²: -3126.5647
  R: 0.1001

SSP5:
  샘플 수: 28
  MAE: 1.0154
  RMSE: 1.6239
  R²: -151351.3457
  R: 0.1967


In [32]:
# 예측 결과 저장
predictions_df = pd.DataFrame(y_pred_final, columns=[f'pred_{col}' for col in target_names])
actuals_df = pd.DataFrame(y_test, columns=[f'actual_{col}' for col in target_names])
scenario_df = pd.DataFrame({'scenario': scenario_test})

In [33]:
results_df = pd.concat([scenario_df, actuals_df, predictions_df], axis=1)
results_df.to_csv('cnidium_predictions.csv', index=False)
print("✓ 예측 결과 저장: cnidium_predictions.csv")

✓ 예측 결과 저장: cnidium_predictions.csv


In [34]:
# 특성 중요도 저장
importance_df.to_csv('feature_importance.csv', index=False)
print("✓ 특성 중요도 저장: feature_importance.csv")

✓ 특성 중요도 저장: feature_importance.csv


In [35]:
# 모델 성능 저장
final_results.to_csv('model_performance.csv', index=False)
print("✓ 모델 성능 저장: model_performance.csv")

✓ 모델 성능 저장: model_performance.csv
